# JEV S6E9 â€” exact-value memory candidate


This is a reproducible, competition-only candidate for `playground-series-s6e9`. 
It trains from the attached competition `train.csv` and generates `submission.csv` 
for the attached `test.csv`; it does not read predictions or labels from other participants.


The feature design is an independent implementation of the publicly documented 
exact-value target-encoding and high-resolution LightGBM idea. It is retained as a 
research candidate and must pass the local schema audit before submission.


AI disclosure: this notebook was designed with JEV/ Codex using an authenticated ChatGPT 
subscription. The final estimator is trained from scratch here; no OpenAI API is called 
by the notebook and no third-party prediction file is used.


In [ ]:
from pathlib import Path

import json

import time

import numpy as np

import pandas as pd

import lightgbm as lgb

from sklearn.metrics import roc_auc_score

from sklearn.model_selection import KFold, StratifiedKFold


TARGET = 'Will_Buy_EV'

ID = 'id'

SEED = 42

N_FOLDS = 5

TE_ALPHA = 1.0

TE_COLS = ['Annual_Income_USD', 'Daily_Commute_km', 'income_100', 'income_1000', 'commute_km', 'income_nearest_original']

ROOTS = [Path('/kaggle/input'), Path('../data'), Path('data')]


def find_one(name):

    hits = []

    for root in ROOTS:

        if root.exists(): hits.extend(sorted(root.rglob(name)))

    if len(hits) != 1: raise RuntimeError(f'{name}: expected one file, found {hits}')

    return hits[0]


train = pd.read_csv(find_one('train.csv'))

test = pd.read_csv(find_one('test.csv'))

sample = pd.read_csv(find_one('sample_submission.csv'))

y = (train[TARGET] == 'Yes').astype('int8').to_numpy()

assert train.shape[0] == 668665 and test.shape[0] == 286571

assert sample.columns.tolist() == [ID, TARGET]

assert test[ID].equals(sample[ID])

source_hits = sorted(Path('/kaggle/input').rglob('EV_Adoption_and_Range_Anxiety*.csv')) if Path('/kaggle/input').exists() else []

source_income = np.sort(pd.read_csv(source_hits[0])['Annual_Income_USD'].dropna().unique()) if source_hits else np.sort(train['Annual_Income_USD'].dropna().unique())

def nearest_original(values):

    values = np.asarray(values, dtype='float64')

    pos = np.clip(np.searchsorted(source_income, values), 1, len(source_income)-1)

    left, right = source_income[pos-1], source_income[pos]

    return np.where(np.abs(values-left) <= np.abs(right-values), left, right).astype('float32')

print(train.shape, test.shape, y.mean(), 'source incomes', len(source_income))


## Target-free features


The raw fields are retained. Numeric identity keys are added only as target-free 
quantisations; their target statistics are computed inside each outer fold and, for 
training rows, inside an inner fold so a row never sees its own label.


In [ ]:
CATS = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

FEATURES = [c for c in test.columns if c != ID]

X = train[FEATURES].copy()

XT = test[FEATURES].copy()

for c in CATS:

    cats = pd.Categorical(pd.concat([X[c], XT[c]], ignore_index=True)).categories

    X[c] = pd.Categorical(X[c], categories=cats)

    XT[c] = pd.Categorical(XT[c], categories=cats)

for frame in (X, XT):

    frame['income_100'] = (frame['Annual_Income_USD'] // 100).astype('float32')

    frame['income_1000'] = (frame['Annual_Income_USD'] // 1000).astype('float32')

    frame['commute_km'] = np.floor(frame['Daily_Commute_km']).astype('float32')

    frame['income_nearest_original'] = nearest_original(frame['Annual_Income_USD'].to_numpy())

    income_int = frame['Annual_Income_USD'].fillna(-1).round().astype('int64')

    for digit in range(6):

        frame[f'income_digit_{digit}'] = ((income_int // (10 ** digit)) % 10).astype('int8')

    frame['total_charging'] = frame['Charging_Stations_Near_Home'] + frame['Charging_Stations_Near_Work']

print('features', X.shape[1], TE_COLS)


In [ ]:
def cross_fit_te(xtr, ytr, xva, xte, cols, alpha=1.0, inner=5):

    prior = float(ytr.mean())

    out_tr = {}

    out_va = {}

    out_te = {}

    for col in cols:

        tr_key = xtr[col].reset_index(drop=True)

        va_key = xva[col].reset_index(drop=True)

        te_key = xte[col].reset_index(drop=True)

        tr_enc = np.full(len(tr_key), prior, dtype='float32')

        kf = KFold(inner, shuffle=True, random_state=SEED)

        for a, b in kf.split(tr_key):

            stats = pd.DataFrame({'key': tr_key.iloc[a].to_numpy(), 'y': ytr[a]}).groupby('key')['y'].agg(['sum', 'size'])

            mapping = (stats['sum'] + alpha * prior) / (stats['size'] + alpha)

            tr_enc[b] = tr_key.iloc[b].map(mapping).fillna(prior).to_numpy('float32')

        stats = pd.DataFrame({'key': tr_key.to_numpy(), 'y': ytr}).groupby('key')['y'].agg(['sum', 'size'])

        mapping = (stats['sum'] + alpha * prior) / (stats['size'] + alpha)

        out_tr['te_' + col] = tr_enc

        out_va['te_' + col] = va_key.map(mapping).fillna(prior).to_numpy('float32')

        out_te['te_' + col] = te_key.map(mapping).fillna(prior).to_numpy('float32')

    return pd.DataFrame(out_tr), pd.DataFrame(out_va), pd.DataFrame(out_te)


def add_te(frame, enc):

    out = frame.reset_index(drop=True).copy()

    for col in enc.columns: out[col] = enc[col].to_numpy()

    return out


## Fold-safe training


The parameters are frozen before execution: 5 stratified folds, `max_bin=4095`, 
shallow high-resolution LightGBM, early stopping, and test prediction averaged across folds.


In [ ]:
folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

oof = np.zeros(len(train), dtype='float32')

pred = np.zeros(len(test), dtype='float64')

fold_scores = []

best_iters = []

started = time.perf_counter()

for fold, (fit_idx, val_idx) in enumerate(folds.split(X, y)):

    x_fit = X.iloc[fit_idx].copy()

    x_val = X.iloc[val_idx].copy()

    x_te = XT.copy()

    e_fit, e_val, e_te = cross_fit_te(x_fit[TE_COLS], y[fit_idx], x_val[TE_COLS], x_te[TE_COLS], TE_COLS, TE_ALPHA)

    x_fit = add_te(x_fit, e_fit)

    x_val = add_te(x_val, e_val)

    x_te = add_te(x_te, e_te)

    model = lgb.LGBMClassifier(objective='binary', metric='auc', learning_rate=0.02, n_estimators=50000,

        num_leaves=32, max_depth=5, min_child_samples=10, colsample_bytree=0.3, subsample=0.8,

        subsample_freq=1, reg_lambda=2.0, reg_alpha=0.071, max_bin=4095, random_state=SEED + fold,

        n_jobs=-1, verbosity=-1)

    model.fit(x_fit, y[fit_idx], categorical_feature=CATS, eval_set=[(x_val, y[val_idx])],

        callbacks=[lgb.early_stopping(200, verbose=False)])

    oof[val_idx] = model.predict_proba(x_val, num_iteration=model.best_iteration_)[:, 1]

    pred += model.predict_proba(x_te, num_iteration=model.best_iteration_)[:, 1] / N_FOLDS

    score = roc_auc_score(y[val_idx], oof[val_idx])

    fold_scores.append(float(score)); best_iters.append(int(model.best_iteration_))

    print(f'fold {fold}: auc={score:.7f}, trees={model.best_iteration_}', flush=True)

auc = float(roc_auc_score(y, oof))

print({'oof_auc': auc, 'fold_scores': fold_scores, 'best_iters': best_iters, 'elapsed_s': time.perf_counter()-started})


In [ ]:
submission = pd.DataFrame({ID: test[ID], TARGET: pred})

assert submission.columns.tolist() == [ID, TARGET]

assert submission[ID].equals(sample[ID])

assert len(submission) == len(test) and np.isfinite(submission[TARGET]).all()

submission.to_csv('/kaggle/working/submission.csv', index=False)

pd.DataFrame({ID: train[ID], 'fold': -1, TARGET: y, 'oof_prediction': oof}).to_csv('/kaggle/working/oof.csv', index=False)

Path('/kaggle/working/report.json').write_text(json.dumps({'oof_auc': auc, 'fold_scores': fold_scores, 'best_iters': best_iters}, indent=2))

print('wrote /kaggle/working/submission.csv')


In [ ]:

# Independent multiscale exact-key TE ablation. The prior v3 cell trains the
# validated candidate; this cell trains a second model from the same raw data
# and selects only an OOF-supported rank blend.
M = X.copy()
MT = XT.copy()
for frame in (M, MT):
    frame['recipe_score'] = (
        1.2 * frame['Annual_Income_USD'].astype('float64') / 100000.0
        + 0.6 * frame['Environmental_Concern_Level'].astype('float64')
        + 2.0 * (frame['Subsidy_Available'] == 'Yes').astype('float64')
        - 1.0 * (frame['Range_Anxiety_Level'] == 'Medium').astype('float64')
        - 3.0 * (frame['Range_Anxiety_Level'] == 'High').astype('float64')
    ).astype('float32')
    home = (frame['Home_Charging_Possible'] == 'Yes').astype('float32')
    frame['worry_score'] = (
        frame['Daily_Commute_km']
        - 5.0 * frame['Charging_Stations_Near_Home']
        - 5.0 * frame['Charging_Stations_Near_Work']
        - 150.0 * home
    ).astype('float32')
    frame['chargers_total'] = (
        frame['Charging_Stations_Near_Home'] + frame['Charging_Stations_Near_Work']
    ).astype('float32')
    frame['income_x_subsidy'] = (
        frame['Annual_Income_USD'] / 100000.0 * (frame['Subsidy_Available'] == 'Yes')
    ).astype('float32')
    frame['concern_x_subsidy'] = (
        frame['Environmental_Concern_Level'] * (frame['Subsidy_Available'] == 'Yes')
    ).astype('float32')
    income_i = frame['Annual_Income_USD'].round().astype('int64')
    commute_i = frame['Daily_Commute_km'].round().astype('int64')
    frame['income_exact_int'] = income_i.astype(str)
    frame['income100_floor'] = (income_i // 100).astype(str)
    frame['income1000_floor'] = (income_i // 1000).astype(str)
    frame['commute_integer'] = commute_i.astype(str)

KEY_COLS = [
    'Annual_Income_USD', 'Daily_Commute_km', 'income_exact_int',
    'income100_floor', 'income1000_floor', 'commute_integer',
]
for col in ['income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer']:
    categories = pd.Categorical(pd.concat([M[col], MT[col]], ignore_index=True)).categories
    M[col] = pd.Categorical(M[col], categories=categories)
    MT[col] = pd.Categorical(MT[col], categories=categories)

def multi_te(xtr, ytr, xva, xte, cols, alpha, prefix):
    a, b, c = cross_fit_te(xtr[cols], ytr, xva[cols], xte[cols], cols, alpha)
    a.columns = [prefix + col for col in a.columns]
    b.columns = [prefix + col for col in b.columns]
    c.columns = [prefix + col for col in c.columns]
    return a, b, c

folds_ms = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof_ms = np.zeros(len(train), dtype='float32')
pred_ms = np.zeros(len(test), dtype='float64')
ms_scores = []
ms_iters = []
ms_cats = CATS + ['income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer']
ms_base_cols = list(M.columns)
ms_te_cols = [f'{prefix}te_{col}' for prefix in ['te1_', 'te10_', 'te100_'] for col in KEY_COLS]
ms_columns = ms_base_cols + ms_te_cols
for fold, (fit_idx, val_idx) in enumerate(folds_ms.split(M, y)):
    x_fit = M.iloc[fit_idx].copy()
    x_val = M.iloc[val_idx].copy()
    x_te = MT.copy()
    enc_parts = []
    for alpha, prefix in [(1.0, 'te1_'), (10.0, 'te10_'), (100.0, 'te100_')]:
        enc_parts.append(multi_te(x_fit, y[fit_idx], x_val, x_te, KEY_COLS, alpha, prefix))
    for index, frame in enumerate((x_fit, x_val, x_te)):
        for tr_enc, va_enc, te_enc in enc_parts:
            enc = (tr_enc, va_enc, te_enc)[index]
            for col in enc.columns:
                frame[col] = enc[col].to_numpy()
    model_ms = lgb.LGBMClassifier(
        objective='binary', metric='auc', learning_rate=0.015, n_estimators=50000,
        num_leaves=32, max_depth=5, min_child_samples=30, colsample_bytree=0.65,
        subsample=0.8, subsample_freq=1, reg_lambda=1.0, reg_alpha=0.5,
        max_bin=1024, random_state=SEED + fold, n_jobs=-1, verbosity=-1,
    )
    model_ms.fit(
        x_fit[ms_columns], y[fit_idx], categorical_feature=ms_cats,
        eval_set=[(x_val[ms_columns], y[val_idx])],
        callbacks=[lgb.early_stopping(300, verbose=False)],
    )
    oof_ms[val_idx] = model_ms.predict_proba(x_val[ms_columns], num_iteration=model_ms.best_iteration_)[:, 1]
    pred_ms += model_ms.predict_proba(x_te[ms_columns], num_iteration=model_ms.best_iteration_)[:, 1] / N_FOLDS
    ms_scores.append(float(roc_auc_score(y[val_idx], oof_ms[val_idx])))
    ms_iters.append(int(model_ms.best_iteration_))
    print(f'multiscale fold {fold}: auc={ms_scores[-1]:.7f}, trees={ms_iters[-1]}', flush=True)

def rank_ms(values):
    order = np.argsort(values, kind='mergesort')
    ranks = np.empty(len(values), dtype='float64')
    ranks[order] = np.arange(len(values), dtype='float64')
    return (ranks + 0.5) / len(values)

best = (-1.0, 0.0)
for weight in np.linspace(0.0, 1.0, 21):
    candidate = weight * rank_ms(oof_ms) + (1.0 - weight) * rank_ms(oof)
    score = float(roc_auc_score(y, candidate))
    if score > best[0]:
        best = (score, float(weight))
weight_ms = best[1]
blend_oof_ms = weight_ms * rank_ms(oof_ms) + (1.0 - weight_ms) * rank_ms(oof)
blend_test_ms = weight_ms * rank_ms(pred_ms) + (1.0 - weight_ms) * rank_ms(pred)

def pure_ms(frame, values):
    income = frame['Annual_Income_USD'].to_numpy(dtype='float64')
    commute = frame['Daily_Commute_km'].to_numpy(dtype='float64')
    env = frame['Environmental_Concern_Level'].to_numpy(dtype='float64')
    subsidy = (frame['Subsidy_Available'] == 'Yes').to_numpy()
    medium = (frame['Range_Anxiety_Level'] == 'Medium').to_numpy()
    high = (frame['Range_Anxiety_Level'] == 'High').to_numpy()
    out = values.copy()
    out[income >= 170537.0] += 10.0
    out[(income >= 31004.0) & (income <= 41970.0)] -= 10.0
    out[commute >= 83.0] -= 5.0
    out[(income == 30000.0) & (~subsidy) & ((env == 1.0) | medium | high)] -= 5.0
    return rank_ms(out)

final_oof_ms = pure_ms(train, blend_oof_ms)
final_pred_ms = pure_ms(test, blend_test_ms)
report_ms = {
    'oof_auc_multiscale': float(roc_auc_score(y, oof_ms)),
    'oof_auc_rank_blend': float(roc_auc_score(y, blend_oof_ms)),
    'oof_auc_final': float(roc_auc_score(y, final_oof_ms)),
    'blend_weight_multiscale': weight_ms,
    'fold_scores': ms_scores,
    'best_iters': ms_iters,
    'te_alphas': [1.0, 10.0, 100.0],
    'rule_source': 'training-pure deterministic pockets; no external predictions',
}
print(json.dumps(report_ms, indent=2))
submission = pd.DataFrame({ID: test[ID], TARGET: final_pred_ms})
assert submission.columns.tolist() == [ID, TARGET]
assert submission[ID].equals(sample[ID])
assert len(submission) == len(test)
assert np.isfinite(submission[TARGET]).all() and submission[TARGET].between(0.0, 1.0).all()
assert submission[TARGET].nunique() == len(test)
submission.to_csv('/kaggle/working/submission.csv', index=False)
pd.DataFrame({ID: train[ID], TARGET: y, 'oof_prediction': final_oof_ms}).to_csv('/kaggle/working/oof.csv', index=False)
Path('/kaggle/working/report.json').write_text(json.dumps(report_ms, indent=2))
print('wrote multiscale /kaggle/working/submission.csv')
